# Synthetic Research Engine — Calibration

Run the calibration backtests and get **real validation numbers** for the
synthetic research engine.

It runs `calibration.py`, which backtests the engine against four datasets
whose answers are already known:

| Backtest | Dataset | Measures |
|---|---|---|
| Choice-conjoint | `choiceData.csv` (72 real human choices) | Does the engine recover real apple-type / freshness / price preferences? |
| Ranking-conjoint | `conjointpizzadata.csv` | Rank correlation vs a known 1-16 ranking |
| Optimism | `AB_Test_Results.csv` | Does the panel wrongly favour the "new" variant? |
| Ignorance | `abtest1.csv` | Given undescribed options, does it stay near-uniform? |

Run the cells top to bottom. Total time: ~2-4 minutes (~12 LLM calls).

### 1. Install dependencies

In [ ]:
!pip install -q anthropic openai google-generativeai pandas numpy scipy

### 2. Get the project code (clones the GitHub repo)

In [ ]:
import os
if os.path.isdir('/content/dashboard'):
    !cd /content/dashboard && git pull -q
else:
    !git clone -q https://github.com/joongsukkie/dashboard.git /content/dashboard
%cd /content/dashboard
# Confirm the calibration datasets are present.
!ls -la AB_Test_Results.csv abtest1.csv conjointpizzadata.csv choiceData.csv

### 3. Your AI key

Pick the provider you have a key for, then run the cell and paste the key
when prompted (it stays hidden).

**Use the same provider/model your live app uses** — otherwise the
calibration numbers won't reflect the deployed engine.

In [ ]:
import getpass

PROVIDER = "anthropic"   # "anthropic" | "openai" | "gemini"

API_KEY = getpass.getpass(f"Paste your {PROVIDER} API key (hidden): ").strip()
print(f"Key captured for {PROVIDER} ({len(API_KEY)} chars).")

### 4. Define the LLM caller

These match the callers in `app.py` so the notebook exercises the exact
same engine the live app uses. If a model name is ever deprecated, edit
the `MODELS` dict below.

In [ ]:
MODELS = {
    "openai":    "gpt-4o",
    "anthropic": "claude-sonnet-4-20250514",
    "gemini":    "gemini-1.5-pro",
}

def call_openai(api_key, prompt, strict=False):
    from openai import OpenAI
    client = OpenAI(api_key=api_key)
    system = "You are a data analytics expert. Return only valid JSON."
    if strict:
        system += " Return ONLY a JSON object, no other text, no markdown, no fences."
    resp = client.chat.completions.create(
        model=MODELS["openai"],
        messages=[{"role": "system", "content": system},
                  {"role": "user", "content": prompt}],
        temperature=0.2,
        response_format={"type": "json_object"},
    )
    return resp.choices[0].message.content

def call_anthropic(api_key, prompt, strict=False):
    import anthropic
    client = anthropic.Anthropic(api_key=api_key)
    system = "You are a data analytics expert. Return only valid JSON, no markdown fences, no prose."
    if strict:
        system += " CRITICAL: Return ONLY a JSON object starting with { and ending with }."
    resp = client.messages.create(
        model=MODELS["anthropic"], max_tokens=4096, system=system,
        messages=[{"role": "user", "content": prompt}],
    )
    return resp.content[0].text

def call_gemini(api_key, prompt, strict=False):
    import google.generativeai as genai
    genai.configure(api_key=api_key)
    system = "You are a data analytics expert. Return only valid JSON with no markdown fences."
    if strict:
        system += " Return ONLY a JSON object. No prose. No code fences."
    model = genai.GenerativeModel(
        MODELS["gemini"], system_instruction=system,
        generation_config={"response_mime_type": "application/json", "temperature": 0.2},
    )
    return model.generate_content(prompt).text

CALLERS = {"openai": call_openai, "anthropic": call_anthropic, "gemini": call_gemini}
caller = CALLERS[PROVIDER]
print(f"Caller ready: {PROVIDER} -> {MODELS[PROVIDER]}")

### 5. Run the calibration

This runs all four backtests. You'll see progress lines, then the report.

In [ ]:
import calibration

profile = calibration.run_calibration(caller, API_KEY)   # writes calibration_profile.json
calibration._print_report(profile)

### 6. Per-backtest detail

In [ ]:
import json
print("TRUST GRADES        :", profile.get("trust"))
print("RECOMMENDED FIXES   :", profile.get("recommended_corrections"))
print()
for r in profile.get("results", []):
    print("-" * 70)
    print(r["case"])
    print("  context:", r["note"])
    print("  scores :", {k: v for k, v in r["score"].items()
                          if k not in ("ok", "predicted_order")})

### 7. Download the calibration profile

`calibration_profile.json` is what the live app reads to attach an
honest trust level to every synthetic result. Download it, then commit
it to your repo (or upload it to the server) so the deployed engine
picks it up.

In [ ]:
from google.colab import files
files.download("calibration_profile.json")

### How to read the numbers

**Choice-conjoint** (the strongest test — real discrete-choice experiment)
- `type_spearman` / `freshness_spearman` — rank correlation with real human
  preference. `1.0` = perfect order, `0` = no relationship, negative = backwards.
- `price_sign_correct` — did the engine prefer the cheaper option? (should be `True`)
- `top_type_correct` / `top_freshness_correct` — did it pick the real favourite?

**Comparison**
- `optimism_bias` — how much the panel over-favours the "new" option. The
  AB test's control actually won, so a positive number = optimism bias.
- `ignorance_overconfidence` — distance from a uniform split on undescribed
  options. Near `0` = well-calibrated; large = it invents signal.

**Trust grades** flow into the live app: a study's confidence is capped at
the grade its backtest family earned, so the engine can never claim more
certainty than it has proven.

A good headline result looks like: *"recovered real apple-type preference
at Spearman 0.8, read price direction correctly, optimism bias +0.1."*